In [ ]:
import os

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from langchain.llms import HuggingFacePipeline
from transformers import pipeline, AutoTokenizer
import pandas as pd
from sentence_transformers import SentenceTransformer
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.schema import Document
from sentence_transformers import SentenceTransformer


c:\Users\LeungSt\AppData\Local\miniconda3\envs\pmadvisor\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def row_to_document(row, text_cols=None):
    """
    Convert a pandas Series (row) into a LangChain document with:
    - page_content from selected text columns
    - metadata from remaining columns
    """
    if text_cols is None:
        text_cols = []

    # Build content string from specified text columns
    content_lines = []
    for col in text_cols:
        val = row.get(col)
        if pd.notnull(val):
            content_lines.append(f"{col}: {val}")
    page_content = "\n".join(content_lines)

    # All other columns become metadata
    metadata = {}
    for col in row.index:
        if col not in text_cols:
            val = row[col]
            if pd.notnull(val):
                metadata[col] = val

    return {
        "page_content": page_content,
        "metadata": metadata
    }



In [ ]:
# Load the Excel file
# Get all file names in the "data" folder
data_folder = 'data'
file_names = [f for f in os.listdir(data_folder) if f.endswith('.xlsx')]
print(f"Found {len(file_names)} .xlsx files in the 'data' folder.")

# Load the local language model with CUDA support
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Read the first .xlsx file
if file_names:
    excel_file_path = os.path.join(data_folder, file_names[0])
    df = pd.read_excel(excel_file_path, header=1)
    print(f"Loaded Excel file: {excel_file_path}")
else:
    print("No .xlsx files found in the 'data' folder.")

# Embedding setup
# embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
# Load model & tokenizer manually on CUDA
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding_tokenizer = AutoTokenizer.from_pretrained(embedding_model_name)
# embedding_model = AutoModel.from_pretrained(embedding_model_name).to(device)

# Load using sentence-transformers with GPU
sbert_model = SentenceTransformer(embedding_model_name, device=device)
# ✅ Wrap in LangChain embedding wrapper
embedding = HuggingFaceEmbeddings(model_name=embedding_model_name, 
                                  encode_kwargs={"device": device})

# Define which columns hold semantic text content
text_columns = ['Description of the issue/success', 'Lessons learned', 'Recommendation / Action to be taken for disseminating the issue']

# Apply to entire DataFrame
documents = [row_to_document(row, text_columns) for _, row in df.iterrows()]

# Create LangChain Documents

lc_docs = [Document(page_content=doc["page_content"], metadata=doc["metadata"]) for doc in documents]

# Create vector store (FAISS)
vectorstore = FAISS.from_documents(lc_docs, embedding)

# Initialize retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})



# Load the model
# I tried "EleutherAI/gpt-neo-2.7B" but the answer was not good.
model_id = "HuggingFaceH4/zephyr-7b-alpha"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": torch.cuda.current_device()},
    max_memory={f"cuda:{torch.cuda.current_device()}": "15GiB"}
)

# Create a LLM pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    # device=device,
    max_new_tokens=10000,       # Allow more tokens in the response
    temperature=0.7,            # Add randomness (can adjust)
    top_p=0.95,                 # Top-p (nucleus) sampling
    repetition_penalty=1.1,     # Prevent too much repetition
    pad_token_id=tokenizer.eos_token_id  # Prevent warning
)

llm = HuggingFacePipeline(pipeline=pipe)

# Create a Retrieval QA chain to combine retrieval + generation
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)



# This cell took 25.9 seconds to run last time.




Found 2 .xlsx files in the 'data' folder.
Using device: cuda
Loaded Excel file: data\National lessons-learned database  Project Navigator - Lessons learned_Synthetic.xlsx


C:\Users\LeungSt\AppData\Local\Temp\ipykernel_49988\2599869572.py:29: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name=embedding_model_name,
Loading checkpoint shards: 100%|██████████| 8/8 [00:20<00:00,  2.51s/it]
Device set to use cuda:0
C:\Users\LeungSt\AppData\Local\Temp\ipykernel_49988\2599869572.py:82: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import 

In [ ]:
query = "What lessons were learned regarding stakeholder engagement in infrastructure projects in 2023?"

response = qa_chain({"query": query})

# Output answer
print("Answer:\n", response["result"])

# Output retrieved context (source documents)
print("\nContext Used:")
for doc in response["source_documents"]:
    print("---")
    print("Page Content:\n", doc.page_content)
    print("Metadata:", doc.metadata)
    
# This cell took 3.6 seconds to run last time.

C:\Users\LeungSt\AppData\Local\Temp\ipykernel_49988\3298790301.py:3: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = qa_chain({"query": query})
c:\Users\LeungSt\AppData\Local\miniconda3\envs\pmadvisor\Lib\site-packages\transformers\generation\configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\LeungSt\AppData\Local\miniconda3\envs\pmadvisor\Lib\site-packages\transformers\generation\configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


Answer:
 Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Description of the issue/success: Successful completion of environmental impact study
Lessons learned: Improved data collection methods
Recommendation / Action to be taken for disseminating the issue: Standardize data collection procedures across projects

Description of the issue/success: Successful completion of environmental impact study
Lessons learned: Improved data collection methods
Recommendation / Action to be taken for disseminating the issue: Standardize data collection procedures across projects

Description of the issue/success: Successful completion of environmental impact study
Lessons learned: Improved data collection methods
Recommendation / Action to be taken for disseminating the issue: Standardize data collection procedures across projects

Question: What lessons were learned regarding stakeholde

In [ ]:
query = "What could have been done better to avoid delays due to weather?  What is the average delay due to weather?  What kind of weather would cause the most severe delays?"

response = qa_chain({"query": query})

# Output answer
print("Answer:\n", response["result"])

# Output retrieved context (source documents)
print("\nContext Used:")
for doc in response["source_documents"]:
    print("---")
    print("Page Content:\n", doc.page_content)
    print("Metadata:", doc.metadata)
    
# This cell took 6.6 seconds to run last time.

c:\Users\LeungSt\AppData\Local\miniconda3\envs\pmadvisor\Lib\site-packages\transformers\generation\configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\LeungSt\AppData\Local\miniconda3\envs\pmadvisor\Lib\site-packages\transformers\generation\configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


Answer:
 Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Description of the issue/success: Delay due to unexpected weather conditions
Lessons learned: Improved scheduling and contingency planning
Recommendation / Action to be taken for disseminating the issue: Develop a weather contingency plan for future projects

Description of the issue/success: Issue: Project faced significant delays due to unforeseen site conditions and environmental concerns.
Lessons learned: Lesson Learned: Conducting thorough site assessments and environmental impact studies prior to project initiation can mitigate delays and ensure smoother project execution.
Recommendation / Action to be taken for disseminating the issue: Implement comprehensive site assessments and environmental studies

Description of the issue/success:  Equipment failure caused significant delays and increased costs.
Lessons 

In [ ]:
query = "What is the top reason that caused the longest delays? What is the average delay due to this reason compared with the average delay due to all other reasons?"

response = qa_chain({"query": query})

# Output answer
print("Answer:\n", response["result"])

# Output retrieved context (source documents)
print("\nContext Used:")
for doc in response["source_documents"]:
    print("---")
    print("Page Content:\n", doc.page_content)
    print("Metadata:", doc.metadata)
    
# This cell took 3.5 seconds to run last time.

c:\Users\LeungSt\AppData\Local\miniconda3\envs\pmadvisor\Lib\site-packages\transformers\generation\configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\LeungSt\AppData\Local\miniconda3\envs\pmadvisor\Lib\site-packages\transformers\generation\configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


Answer:
 Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Description of the issue/success: Technical issues delayed system implementation and impacted response times.
Lessons learned: Establishing dedicated technical support teams can mitigate technical issues and ensure timely system implementation.
Recommendation / Action to be taken for disseminating the issue: Establish dedicated technical support teams

Description of the issue/success: Technical issues delayed system implementation and impacted response times.
Lessons learned: Establishing dedicated technical support teams can mitigate technical issues and ensure timely system implementation.
Recommendation / Action to be taken for disseminating the issue: Establish dedicated technical support teams

Description of the issue/success: Technical issues delayed system implementation and impacted response times.
Lessons